# Classical ML

# BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier, SVM, LogisticRegression

In [3]:
import pandas as pd
import numpy as np
import time
import sys
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB  # Import Naive Bayes classifier

# Load the data
df = pd.read_excel('ams_data.xlsx')

# Check for missing values
if df.isnull().sum().any():
    df.fillna(method='ffill', inplace=True)  # Forward fill as an example

# Categorize AMS scores
def categorize_ams(score):
    if 3 <= score <= 5:
        return 'Mild'
    elif 6 <= score <= 9:
        return 'Moderate'
    elif 10 <= score <= 12:
        return 'Severe'
    return 'Unknown'

df['AMS Category'] = df['AMS Total Score'].apply(categorize_ams)
df = df[df['AMS Category'] != 'Unknown']

# Encode categories
label_encoder = LabelEncoder()
df['AMS Encoded'] = label_encoder.fit_transform(df['AMS Category'])

# Feature Engineering: Additional features
df['HR_SpO2_Ratio'] = df['HR (bpm)'] / (df['SpO2 (%)'] + 1e-6)
df['SpO2_HR_Ratio'] = df['SpO2 (%)'] / (df['HR (bpm)'] + 1e-6)
df['HR_Squared'] = df['HR (bpm)'] ** 2
df['SpO2_Squared'] = df['SpO2 (%)'] ** 2

# Prepare features and labels
X = df[['HR (bpm)', 'SpO2 (%)', 'HR_SpO2_Ratio', 'SpO2_HR_Ratio', 'HR_Squared', 'SpO2_Squared']]
y = df['AMS Encoded']

# Dynamic oversampling
df_mild = df[df['AMS Category'] == 'Mild']
df_moderate = df[df['AMS Category'] == 'Moderate']
df_severe = df[df['AMS Category'] == 'Severe']

max_size = max(len(df_mild), len(df_moderate), len(df_severe))

# Upsample each class
df_mild_upsampled = resample(df_mild, replace=True, n_samples=max_size, random_state=40)
df_moderate_upsampled = resample(df_moderate, replace=True, n_samples=max_size, random_state=40)
df_severe_upsampled = resample(df_severe, replace=True, n_samples=max_size, random_state=40)

df_combined = pd.concat([df_mild_upsampled, df_moderate_upsampled, df_severe_upsampled])
X_resampled = df_combined[['HR (bpm)', 'SpO2 (%)', 'HR_SpO2_Ratio', 'SpO2_HR_Ratio', 'HR_Squared', 'SpO2_Squared']]
y_resampled = df_combined['AMS Encoded']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.3, random_state=40
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Algorithm dictionary to store metrics for each model
models = {
    'Random Forest': RandomForestClassifier(n_estimators=2, max_depth=1, random_state=42),
    'Bagging': BaggingClassifier(n_estimators=2, random_state=42),  # Default base_estimator is DecisionTreeClassifier
    'Logistic Regression': LogisticRegression(),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=2, random_state=42),
    'SVM': SVC(kernel='linear', probability=True),
    'Naive Bayes': GaussianNB(),  # Add Naive Bayes classifier
}

# Results dictionary to store analysis for each algorithm
results = {}

# Function to estimate memory usage of models
def calculate_memory(model):
    memory = sys.getsizeof(model)  # Estimate size of the model object itself
    if hasattr(model, 'estimators_'):  # For ensemble models like RandomForest and Bagging
        for estimator in model.estimators_:
            memory += sys.getsizeof(estimator)
            if hasattr(estimator, 'tree_'):
                memory += estimator.tree_.__sizeof__()
    elif hasattr(model, 'support_vectors_'):  # For SVM
        memory += model.support_vectors_.nbytes
    return memory / (1024 * 1024)  # Convert to MB

# Loop over algorithms to fit, predict, and capture performance
for name, model in models.items():  # Changed from algorithms to models
    start_train_time = time.time()
    model.fit(X_train_scaled, y_train)
    end_train_time = time.time()

    # Inference time
    start_inference_time = time.time()
    predictions_train = model.predict(X_train_scaled)
    predictions_test = model.predict(X_test_scaled)
    end_inference_time = time.time()

    # Metrics
    train_time = end_train_time - start_train_time
    inference_time = end_inference_time - start_inference_time
    train_acc = accuracy_score(y_train, predictions_train) * 100
    test_acc = accuracy_score(y_test, predictions_test) * 100
    memory_required = calculate_memory(model)

    # Store results
    results[name] = {
        'Train Accuracy': train_acc,
        'Test Accuracy': test_acc,
        'Training Time': train_time,
        'Inference Time': inference_time,
        'Model Memory (MB)': memory_required
    }

    # Print results
    print(f"\n{name} Results:")
    print("Train Accuracy: {:.2f}%".format(train_acc))
    print("Test Accuracy: {:.2f}%".format(test_acc))
    print("Training Time: {:.4f} seconds".format(train_time))
    print("Inference Time: {:.4f} seconds".format(inference_time))
    print("Model Memory Required: {:.4f} MB".format(memory_required))
    print("\nClassification Report:")
    print(classification_report(y_test, predictions_test, target_names=label_encoder.classes_))


Random Forest Results:
Train Accuracy: 78.57%
Test Accuracy: 77.78%
Training Time: 0.0040 seconds
Inference Time: 0.0000 seconds
Model Memory Required: 0.0003 MB

Classification Report:
              precision    recall  f1-score   support

        Mild       1.00      0.25      0.40         4
    Moderate       0.75      0.86      0.80         7
      Severe       0.78      1.00      0.88         7

    accuracy                           0.78        18
   macro avg       0.84      0.70      0.69        18
weighted avg       0.82      0.78      0.74        18


Bagging Results:
Train Accuracy: 90.48%
Test Accuracy: 83.33%
Training Time: 0.0040 seconds
Inference Time: 0.0000 seconds
Model Memory Required: 0.0003 MB

Classification Report:
              precision    recall  f1-score   support

        Mild       0.67      0.50      0.57         4
    Moderate       0.75      0.86      0.80         7
      Severe       1.00      1.00      1.00         7

    accuracy                     

# HDC

# Pseudo random

In [23]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.utils import resample

# Load the data
df = pd.read_excel('ams_data.xlsx')

# Check for missing values
if df.isnull().sum().any():
    df.fillna(method='ffill', inplace=True)  # Forward fill as an example

# Categorize AMS scores
def categorize_ams(score):
    if 3 <= score <= 5:
        return 'Mild'
    elif 6 <= score <= 9:
        return 'Moderate'
    elif 10 <= score <= 12:
        return 'Severe'
    return 'Unknown'

df['AMS Category'] = df['AMS Total Score'].apply(categorize_ams)
df = df[df['AMS Category'] != 'Unknown']

# Encode categories
label_encoder = LabelEncoder()
df['AMS Encoded'] = label_encoder.fit_transform(df['AMS Category'])

# Feature Engineering: Additional features
df['HR_SpO2_Ratio'] = df['HR (bpm)'] / (df['SpO2 (%)'] + 1e-6)
df['SpO2_HR_Ratio'] = df['SpO2 (%)'] / (df['HR (bpm)'] + 1e-6)
df['HR_Squared'] = df['HR (bpm)'] ** 2
df['SpO2_Squared'] = df['SpO2 (%)'] ** 2

# Prepare features and labels
X = df[['HR (bpm)', 'SpO2 (%)', 'HR_SpO2_Ratio', 'SpO2_HR_Ratio', 'HR_Squared', 'SpO2_Squared']]
y = df['AMS Encoded']

# Dynamic oversampling
df_mild = df[df['AMS Category'] == 'Mild']
df_moderate = df[df['AMS Category'] == 'Moderate']
df_severe = df[df['AMS Category'] == 'Severe']

max_size = max(len(df_mild), len(df_moderate), len(df_severe))

# Upsample each class
df_mild_upsampled = resample(df_mild, replace=True, n_samples=max_size, random_state=40)
df_moderate_upsampled = resample(df_moderate, replace=True, n_samples=max_size, random_state=40)
df_severe_upsampled = resample(df_severe, replace=True, n_samples=max_size, random_state=40)

df_combined = pd.concat([df_mild_upsampled, df_moderate_upsampled, df_severe_upsampled])
X_resampled = df_combined[['HR (bpm)', 'SpO2 (%)', 'HR_SpO2_Ratio', 'SpO2_HR_Ratio', 'HR_Squared', 'SpO2_Squared']]
y_resampled = df_combined['AMS Encoded']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.3, random_state=40
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Hyperdimensional Computing (HDC) parameters
D = 100  # Experiment with different values
NUM_CLASSES = len(np.unique(y_train))
NUM_SAMPLES = X_train_scaled.shape[0]

# Create a random projection matrix
np.random.seed(40)  # For reproducibility
proj = np.random.randn(X_train_scaled.shape[1], D)

def project_data(data, proj):
    return np.dot(data, proj)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = project_data(X_train_scaled, proj)

# Create class hypervectors
class_hypervectors = np.zeros((NUM_CLASSES, D))
for i in range(NUM_SAMPLES):
    class_hypervectors[y_train.iloc[i]] += X_train_proj[i]

def classify(images, class_hypervectors):
    similarities = cosine_similarity(images, class_hypervectors)
    classifications = np.argmax(similarities, axis=1)
    return classifications

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = classify(X_train_proj, class_hypervectors)
acc_train = accuracy_score(y_train, predictions_train) * 100
print("HDC (Cosine Similarity) Train Accuracy: ", acc_train)

# Measure inference time
start_inference_time = time.time()

# Project test data to hyperdimensional space
X_test_proj = project_data(X_test_scaled, proj)

# Classify test data
predictions_test = classify(X_test_proj, class_hypervectors)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
acc_test = accuracy_score(y_test, predictions_test) * 100
print("HDC (Cosine Similarity) Test Accuracy: ", acc_test)

# Calculate model memory requirement
model_memory = proj.nbytes + class_hypervectors.nbytes

print("HDC (Cosine Similarity) Training Time: {:.4f} seconds".format(training_time))
print("HDC (Cosine Similarity) Inference Time: {:.4f} seconds".format(inference_time))
print("HDC (Cosine Similarity) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

# Display classification report
print(classification_report(y_test, predictions_test, target_names=label_encoder.classes_))

HDC (Cosine Similarity) Train Accuracy:  80.95238095238095
HDC (Cosine Similarity) Test Accuracy:  77.77777777777779
HDC (Cosine Similarity) Training Time: 0.0000 seconds
HDC (Cosine Similarity) Inference Time: 0.0010 seconds
HDC (Cosine Similarity) Model Memory Required: 0.0069 MB
              precision    recall  f1-score   support

        Mild       0.50      0.25      0.33         4
    Moderate       0.67      0.86      0.75         7
      Severe       1.00      1.00      1.00         7

    accuracy                           0.78        18
   macro avg       0.72      0.70      0.69        18
weighted avg       0.76      0.78      0.75        18



# Quasi random

In [21]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.utils import resample
from sobol_seq import i4_sobol_generate

# Load the data
df = pd.read_excel('ams_data.xlsx')

# Check for missing values
if df.isnull().sum().any():
    df.fillna(method='ffill', inplace=True)  # Forward fill as an example

# Categorize AMS scores
def categorize_ams(score):
    if 3 <= score <= 5:
        return 'Mild'
    elif 6 <= score <= 9:
        return 'Moderate'
    elif 10 <= score <= 12:
        return 'Severe'
    return 'Unknown'

df['AMS Category'] = df['AMS Total Score'].apply(categorize_ams)
df = df[df['AMS Category'] != 'Unknown']

# Encode categories
label_encoder = LabelEncoder()
df['AMS Encoded'] = label_encoder.fit_transform(df['AMS Category'])

# Feature Engineering: Additional features
df['HR_SpO2_Ratio'] = df['HR (bpm)'] / (df['SpO2 (%)'] + 1e-6)
df['SpO2_HR_Ratio'] = df['SpO2 (%)'] / (df['HR (bpm)'] + 1e-6)
df['HR_Squared'] = df['HR (bpm)'] ** 2
df['SpO2_Squared'] = df['SpO2 (%)'] ** 2

# Prepare features and labels
X = df[['HR (bpm)', 'SpO2 (%)', 'HR_SpO2_Ratio', 'SpO2_HR_Ratio', 'HR_Squared', 'SpO2_Squared']]
y = df['AMS Encoded']

# Dynamic oversampling
df_mild = df[df['AMS Category'] == 'Mild']
df_moderate = df[df['AMS Category'] == 'Moderate']
df_severe = df[df['AMS Category'] == 'Severe']

max_size = max(len(df_mild), len(df_moderate), len(df_severe))

# Upsample each class
df_mild_upsampled = resample(df_mild, replace=True, n_samples=max_size, random_state=40)
df_moderate_upsampled = resample(df_moderate, replace=True, n_samples=max_size, random_state=40)
df_severe_upsampled = resample(df_severe, replace=True, n_samples=max_size, random_state=40)

df_combined = pd.concat([df_mild_upsampled, df_moderate_upsampled, df_severe_upsampled])
X_resampled = df_combined[['HR (bpm)', 'SpO2 (%)', 'HR_SpO2_Ratio', 'SpO2_HR_Ratio', 'HR_Squared', 'SpO2_Squared']]
y_resampled = df_combined['AMS Encoded']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.3, random_state=40
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Hyperdimensional Computing (HDC) parameters
D = 100  # Experiment with different values
NUM_CLASSES = len(np.unique(y_train))
NUM_SAMPLES = X_train_scaled.shape[0]

# Create a Sobol sequence projection matrix
proj = i4_sobol_generate(X_train_scaled.shape[1], D)

def project_data(data, proj):
    return np.dot(data, proj.T)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = project_data(X_train_scaled, proj)

# Create class hypervectors
class_hypervectors = np.zeros((NUM_CLASSES, D))
for i in range(NUM_SAMPLES):
    class_hypervectors[y_train.iloc[i]] += X_train_proj[i]

def classify(images, class_hypervectors):
    similarities = cosine_similarity(images, class_hypervectors)
    classifications = np.argmax(similarities, axis=1)
    return classifications

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = classify(X_train_proj, class_hypervectors)
acc_train = accuracy_score(y_train, predictions_train) * 100
print("HDC (Cosine Similarity) Train Accuracy: ", acc_train)

# Measure inference time
start_inference_time = time.time()

# Project test data to hyperdimensional space
X_test_proj = project_data(X_test_scaled, proj)

# Classify test data
predictions_test = classify(X_test_proj, class_hypervectors)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
acc_test = accuracy_score(y_test, predictions_test) * 100
print("HDC (Cosine Similarity) Test Accuracy: ", acc_test)

# Calculate model memory requirement
model_memory = proj.nbytes + class_hypervectors.nbytes

print("HDC (Cosine Similarity) Training Time: {:.4f} seconds".format(training_time))
print("HDC (Cosine Similarity) Inference Time: {:.4f} seconds".format(inference_time))
print("HDC (Cosine Similarity) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

# Display classification report
print(classification_report(y_test, predictions_test, target_names=label_encoder.classes_))

HDC (Cosine Similarity) Train Accuracy:  80.95238095238095
HDC (Cosine Similarity) Test Accuracy:  77.77777777777779
HDC (Cosine Similarity) Training Time: 0.0000 seconds
HDC (Cosine Similarity) Inference Time: 0.0000 seconds
HDC (Cosine Similarity) Model Memory Required: 0.0069 MB
              precision    recall  f1-score   support

        Mild       0.50      0.25      0.33         4
    Moderate       0.67      0.86      0.75         7
      Severe       1.00      1.00      1.00         7

    accuracy                           0.78        18
   macro avg       0.72      0.70      0.69        18
weighted avg       0.76      0.78      0.75        18

